In [1]:
import os
while not os.path.isdir("DataTypes"):
    os.chdir("..")

import json
import multiprocessing

load("DataTypes/Grafo2Terminales.sage")
load("Scripts/NumeroDe2Bosques.sage")
load("Scripts/GrafoBalloon.sage")
load("Scripts/CantidadesEstructurales.sage")

In [2]:
INICIO, FIN = 7, 10      # rango de n (ambos inclusive), con los mismos m que la búsqueda de maximizantes
NUM_HILOS = 16           # procesos en paralelo
NUM_PARTES = 4 * NUM_HILOS   # cada (n, m) se parte en NUM_PARTES trozos de geng (res/mod) para balancear carga
ARCHIVO_SUPERADORES = "Datos/SuperadoresDelGlobo.json"

In [3]:
def GrafosConexosParte(n, m, res, mod):
    """
        Parte res/mod (0 <= res < mod) de los grafos conexos de n vértices y
        m aristas; la unión de las mod partes es exactamente GenerarGrafosConexos(n, m)
        y las partes son disjuntas (opción res/mod de geng).
    """
    return graphs.nauty_geng(f"{n} -c {m}:{m} {res}/{mod}")

def SuperadoresEnParte(args):
    """
        Trabajo de un proceso: recorre la parte res/mod de C_{n,m} y todas las
        ubicaciones de 2 terminales, y retorna la lista de pares (G, {s,t}) con
        t2 estrictamente mayor que t2_globo, salvo isomorfismo que fija s,t.
    """
    n, m, res, mod, t2_globo = args
    encontrados, vistos = [], set()
    for G in GrafosConexosParte(n, m, res, mod):
        V = G.vertices()
        for i in range(n):
            for j in range(i + 1, n):
                TC = Grafo2Terminales(G, V[i], V[j])
                valor = NumeroDe2Bosques(TC)
                if valor <= t2_globo:
                    continue
                clave = G.canonical_label(partition=[[V[i], V[j]], [v for v in V if v not in (V[i], V[j])]]).graph6_string()
                if clave not in vistos:
                    vistos.add(clave)
                    encontrados.append({"s": int(V[i]), "t": int(V[j]), "t2": int(valor),
                                        "graph6": G.graph6_string()})
    return encontrados

def SuperadoresDelGlobo(n, m, num_hilos=NUM_HILOS, num_partes=NUM_PARTES):
    """
        Retorna (t2 del globo, lista de TODOS los pares (G, {s,t}) con t2 > t2(globo),
        ordenada por t2 decreciente). Como las partes de geng son de grafos G distintos
        (no isomorfos), no hay duplicados entre procesos.
    """
    t2_globo = int(NumeroDe2Bosques(ConstruirGrafo2TerminalesBalloon(n, m)))
    trabajos = [(int(n), int(m), res, num_partes, t2_globo) for res in range(num_partes)]
    # fork: los procesos hijos heredan las funciones y librerías cargadas en el notebook
    with multiprocessing.get_context("fork").Pool(num_hilos) as pool:
        partes = pool.map(SuperadoresEnParte, trabajos, chunksize=1)
    superadores = [d for parte in partes for d in parte]
    superadores.sort(key=lambda d: -d["t2"])
    return t2_globo, superadores

In [4]:
def CargarSuperadores(archivo=ARCHIVO_SUPERADORES):
    """
        Lee el archivo y retorna {n: {m: {"t2_globo": int, "superadores": [{"s", "t", "t2", "graph6"}, ...]}}}
        con claves enteras. Para reconstruir un grafo:
        Grafo2Terminales(Graph(d["graph6"]), d["s"], d["t"]).
    """
    if not os.path.exists(archivo):
        return {}
    with open(archivo) as f:
        crudo = json.load(f)
    return {int(n): {int(m): d for m, d in por_m.items()} for n, por_m in crudo.items()}

def GuardarSuperadores(datos, archivo=ARCHIVO_SUPERADORES):
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    tmp = archivo + ".tmp"
    with open(tmp, "w") as f:
        json.dump(datos, f, indent=1)
    os.replace(tmp, archivo)   # escritura atómica: una interrupción no deja el JSON truncado

In [6]:
superadores_por_nm = CargarSuperadores()
for n in range(INICIO, FIN + 1):
    for m in [binomial(n - 2, 2) + 1, binomial(n - 2, 2) + 2]:
        t2_globo, superadores = SuperadoresDelGlobo(n, m)
        mejor = superadores[0]["t2"] if superadores else None
        print(f"n={n}, m={m}: t2(globo) = {t2_globo}, #superadores (salvo isomorfismo) = {len(superadores)}, "
              f"mejor t2 = {mejor}")
        superadores_por_nm.setdefault(int(n), {})[int(m)] = {
            "t2_globo": t2_globo,
            "superadores": superadores,
        }
        GuardarSuperadores(superadores_por_nm)  # se guarda tras cada (n, m) por si se interrumpe

n=7, m=11: t2(globo) = 200, #superadores (salvo isomorfismo) = 61, mejor t2 = 225


KeyboardInterrupt: 